In [1]:
#Data Manipulation specific
import numpy as np
import pandas as pd
import xarray as xr
import netCDF4 as nc
import scipy.stats as stats
from scipy import signal
from netCDF4 import Dataset
#Visualisation specific
import seaborn as sns 
import matplotlib.pyplot as plt
%matplotlib inline

import cdsapi
import xarray as xr
#import rioxarray
from urllib.request import urlopen# start the client
import requests
import certifi
import urllib3
import cdsapi
from shapely.geometry import mapping
import geopandas as gpd

In [ ]:

r = requests.get("http://google.com")       
print(r.status_code)

In [ ]:

http = urllib3.PoolManager(
    cert_reqs='CERT_REQUIRED',
    ca_certs=certifi.where()
)

In [ ]:
cds = cdsapi.Client()# dataset you want to read

In [ ]:
url = 'https://cds.climate.copernicus.eu/api/v2'
key = '183169:0ba5f9d6-870e-4e73-98a2-120ef76c0683'

## hada dkhal l youtube rah ghatl9a kifach dir had lfichier f  lhome dyalk
with open('C:/Users/ASchynO/.cdsapirc', 'w') as f:
    f.write('\n'.join([url, key]))

with open('C:/Users/ASchynO/.cdsapirc') as f:
    print(f.read())

In [ ]:
c = cdsapi.Client()

c.retrieve(
    'reanalysis-era5-single-levels-monthly-means',
    {
        'format': 'netcdf',
        'product_type': 'monthly_averaged_reanalysis_by_hour_of_day',
        'variable': '2m_temperature',
        'year': [
            '1983', '1984', '1985',
            '1986', '1987', '1988',
            '1989', '1990', '1991',
            '1992', '1993', '1994',
            '1995', '1996', '1997',
            '1998', '1999', '2000',
            '2001', '2002', '2003',
            '2004', '2005', '2006',
            '2007', '2008', '2009',
            '2010', '2011', '2012',
            '2013', '2014', '2015',
            '2016', '2017', '2018',
            '2019', '2020', '2021',
            '2022',
        ],
        'month': [
            '01', '02', '03',
            '04', '05', '06',
            '07', '08', '09',
            '10', '11', '12',
        ],
        'time': [
            '04:00', '05:00', '06:00',
        ],
        'area': [
            37, -18, 20.25,
            0,
        ],
    },
    'download.nc')

In [2]:
tmin = xr.open_dataset('D:\PhD\Datasets\ERA5\min.nc')
tmax = xr.open_dataset('D:\PhD\Datasets\ERA5\max.nc')
p = xr.open_dataset('D:\PhD\Datasets\PERSIANN-CDR\CDR.nc')

c:\Users\ASchynO\anaconda3\lib\site-packages\xarray\backends\plugins.py:65: RuntimeWarning: Engine 'rasterio' loading failed:
DLL load failed while importing _base: The specified module could not be found.
  warnings.warn(f"Engine {name!r} loading failed:\n{ex}", RuntimeWarning)


In [90]:
tmax

<xarray.Dataset>
Dimensions:    (longitude: 73, latitude: 68, time: 480)
Coordinates:
  * longitude  (longitude) float32 -18.0 -17.75 -17.5 -17.25 ... -0.5 -0.25 0.0
  * latitude   (latitude) float32 37.0 36.75 36.5 36.25 ... 20.75 20.5 20.25
  * time       (time) datetime64[ns] 1983-01-01T12:00:00 ... 2022-12-01T12:00:00
Data variables:
    t2m        (time, latitude, longitude) float32 15.32 15.3 ... 28.98 28.8
Attributes:
    Conventions:  CF-1.6
    history:      2023-04-03 15:18:57 GMT by grib_to_netcdf-2.25.1: /opt/ecmw...

In [6]:
tmin['t2m'] = (tmin['t2m'] -273.15)
tmax['t2m'] = (tmax['t2m'] -273.15)

In [7]:
p = p.drop_vars('crs')

In [63]:
pdtmin = tmin.to_dataframe(['time', 'latitude', 'longitude'])
pdtmin.columns = ["tmin"]
pdtmax = tmax.to_dataframe(['time', 'latitude', 'longitude'])
pdtmax.columns = ["tmax"]

In [64]:
pdtmin

tmin
time                latitude longitude           
1983-01-01 05:00:00 37.00    -18.00     15.053497
                             -17.75     15.043701
                             -17.50     15.028046
                             -17.25     15.028046
                             -17.00     15.041748
...                                           ...
2022-12-01 05:00:00 20.25    -1.00      16.237488
                             -0.75      16.120056
                             -0.50      15.876099
                             -0.25      16.004608
                              0.00      16.037872

[2382720 rows x 1 columns]

In [65]:
import datetime as dt
index = pdtmin.index
index = pdtmin.index.tolist()
newindex = []
for i in index:
    newindex.append((dt.datetime.strftime(i[0], '%Y-%m-%d'), i[1], i[2]))

pdtmin.index = pd.MultiIndex.from_tuples(newindex, names = ['time', 'lat', 'lon'])
pdtmax.index = pd.MultiIndex.from_tuples(newindex, names = ['time', 'lat', 'lon'])

In [66]:
pdtmin

tmin
time       lat   lon              
1983-01-01 37.00 -18.00  15.053497
                 -17.75  15.043701
                 -17.50  15.028046
                 -17.25  15.028046
                 -17.00  15.041748
...                            ...
2022-12-01 20.25 -1.00   16.237488
                 -0.75   16.120056
                 -0.50   15.876099
                 -0.25   16.004608
                  0.00   16.037872

[2382720 rows x 1 columns]

In [67]:
pdp = p.to_dataframe(['datetime', 'lat', 'lon'])
pdp

precip
datetime   lat  lon              
1983-01-01 37.5 -17.75  49.714588
                -17.50  46.504448
                -17.25  40.851837
                -17.00  37.140987
                -16.75  41.715736
...                           ...
2022-08-01 20.5 -0.75   38.178108
                -0.50   38.342121
                -0.25   35.542309
                 0.00   43.863758
                 0.25   45.698128

[2397612 rows x 1 columns]

In [68]:
index = pdp.index
index = pdp.index.tolist()
newindex = []
for i in index:
    newindex.append((dt.datetime.strftime(i[0], '%Y-%m-%d'), i[1], i[2]))

pdp.index = pd.MultiIndex.from_tuples(newindex, names = ['time', 'lat', 'lon'])

In [69]:
pdp

precip
time       lat  lon              
1983-01-01 37.5 -17.75  49.714588
                -17.50  46.504448
                -17.25  40.851837
                -17.00  37.140987
                -16.75  41.715736
...                           ...
2022-08-01 20.5 -0.75   38.178108
                -0.50   38.342121
                -0.25   35.542309
                 0.00   43.863758
                 0.25   45.698128

[2397612 rows x 1 columns]

In [72]:
result = pd.concat([pdp, pdtmin, pdtmax], axis=1, join='inner')

In [73]:
result
#(37.0, 20.5, -17.75, 0) -> (latxlon)=(67x72) (total of 4824 pixel) lat x 67 + lon

precip       tmin       tmax
time       lat  lon                                    
1983-01-01 37.0 -17.75  45.372795  15.043701  15.295197
                -17.50  41.519978  15.028046  15.289337
                -17.25  38.298519  15.028046  15.287140
                -17.00  33.130142  15.041748  15.273224
                -16.75  33.649406  15.049591  15.265900
...                           ...        ...        ...
2022-08-01 20.5 -1.00   33.327618  29.673553  39.039154
                -0.75   38.178108  29.646149  39.095551
                -0.50   38.342121  29.685303  39.115356
                -0.25   35.542309  29.497437  38.982025
                 0.00   43.863758  29.034912  38.669922

[2296224 rows x 3 columns]

In [75]:
#print(type(pdp.index.get_level_values("datetime")))
dates = pdp.index.get_level_values("time").to_numpy()
dates = list(dict.fromkeys(dates))
#dates = dates[ : -4]
dates

['1983-01-01',
 '1983-02-01',
 '1983-03-01',
 '1983-04-01',
 '1983-05-01',
 '1983-06-01',
 '1983-07-01',
 '1983-08-01',
 '1983-09-01',
 '1983-10-01',
 '1983-11-01',
 '1983-12-01',
 '1984-01-01',
 '1984-02-01',
 '1984-03-01',
 '1984-04-01',
 '1984-05-01',
 '1984-06-01',
 '1984-07-01',
 '1984-08-01',
 '1984-09-01',
 '1984-10-01',
 '1984-11-01',
 '1984-12-01',
 '1985-01-01',
 '1985-02-01',
 '1985-03-01',
 '1985-04-01',
 '1985-05-01',
 '1985-06-01',
 '1985-07-01',
 '1985-08-01',
 '1985-09-01',
 '1985-10-01',
 '1985-11-01',
 '1985-12-01',
 '1986-01-01',
 '1986-02-01',
 '1986-03-01',
 '1986-04-01',
 '1986-05-01',
 '1986-06-01',
 '1986-07-01',
 '1986-08-01',
 '1986-09-01',
 '1986-10-01',
 '1986-11-01',
 '1986-12-01',
 '1987-01-01',
 '1987-02-01',
 '1987-03-01',
 '1987-04-01',
 '1987-05-01',
 '1987-06-01',
 '1987-07-01',
 '1987-08-01',
 '1987-09-01',
 '1987-10-01',
 '1987-11-01',
 '1987-12-01',
 '1988-01-01',
 '1988-02-01',
 '1988-03-01',
 '1988-04-01',
 '1988-05-01',
 '1988-06-01',
 '1988-07-

In [86]:
lisp = [result.loc[d, 'tmax'].to_numpy() for d in dates]
lisp

C:\Users\ASchynO\AppData\Local\Temp\ipykernel_11480\4153118409.py:1: PerformanceWarning: indexing past lexsort depth may impact performance.
  lisp = [result.loc[d, 'tmax'].to_numpy() for d in dates]


[array([15.295197, 15.289337, 15.28714 , ..., 20.820557, 20.902618,
        20.652802], dtype=float32),
 array([14.360382, 14.342804, 14.326691, ..., 25.231628, 25.364227,
        25.159088], dtype=float32),
 array([15.021942 , 15.041718 , 15.0607605, ..., 29.882965 , 29.885162 ,
        29.64853  ], dtype=float32),
 array([14.62854 , 14.630737, 14.640259, ..., 36.22812 , 36.212738,
        36.00174 ], dtype=float32),
 array([15.532562, 15.540619, 15.544281, ..., 40.75348 , 40.82599 ,
        40.698517], dtype=float32),
 array([16.865936, 16.867401, 16.902557, ..., 40.79303 , 40.722687,
        40.486786], dtype=float32),
 array([19.118713, 19.12091 , 19.136292, ..., 41.226013, 41.160065,
        40.915375], dtype=float32),
 array([20.077698, 20.03302 , 20.009552, ..., 40.0289  , 39.933685,
        39.67578 ], dtype=float32),
 array([21.00592 , 21.003723, 21.008118, ..., 39.552704, 39.531464,
        39.338074], dtype=float32),
 array([19.961945, 19.98172 , 20.012482, ..., 35.17682 , 3

In [87]:
stack = np.dstack(lisp)
stack

array([[[15.295197 , 14.360382 , 15.021942 , ..., 18.165588 ,
         19.814697 , 21.220581 ],
        [15.289337 , 14.342804 , 15.041718 , ..., 18.173645 ,
         19.820557 , 21.194916 ],
        [15.28714  , 14.326691 , 15.0607605, ..., 18.191223 ,
         19.841797 , 21.170013 ],
        ...,
        [20.820557 , 25.231628 , 29.882965 , ..., 41.165924 ,
         40.58789  , 39.115356 ],
        [20.902618 , 25.364227 , 29.885162 , ..., 41.015015 ,
         40.48825  , 38.982025 ],
        [20.652802 , 25.159088 , 29.64853  , ..., 40.73001  ,
         40.26187  , 38.66992  ]]], dtype=float32)

In [88]:
def dataframe(p):

    df = pd.DataFrame({'Dates': dates, 'tmax': stack[0][p]})
    df.set_index("Dates", inplace = True)
    return df

In [89]:
dataframe(4823)

,tmax
Dates,
1983-01-01,20.652802
1983-02-01,25.159088
1983-03-01,29.648529
1983-04-01,36.001740
1983-05-01,40.698517
...,...
2022-04-01,35.050079
2022-05-01,38.644287
2022-06-01,40.730011


In [91]:
frames = [dataframe(p) for p in range(0, 4824)]

In [ ]:
frames

In [92]:
df = pd.concat(frames,axis=1)

In [93]:
df

,tmax,tmax,tmax,tmax,tmax,tmax,tmax,tmax,tmax,tmax,...,tmax,tmax,tmax,tmax,tmax,tmax,tmax,tmax,tmax,tmax
Dates,,,,,,,,,,,,,,,,,,,,,
1983-01-01,15.295197,15.289337,15.287140,15.273224,15.265900,15.260040,15.263702,15.256378,15.265900,15.262238,...,19.656433,19.738495,19.847656,20.000031,20.394928,20.550964,20.638885,20.820557,20.902618,20.652802
1983-02-01,14.360382,14.342804,14.326691,14.287872,14.239502,14.186768,14.151611,14.134003,14.123749,14.110565,...,23.770050,23.813293,23.922455,24.049927,24.530518,24.813293,24.965668,25.231628,25.364227,25.159088
1983-03-01,15.021942,15.041718,15.060760,15.062958,15.060760,15.049042,15.035858,14.998505,14.943542,14.928162,...,28.934235,29.037537,29.131317,29.209717,29.554749,29.629486,29.736450,29.882965,29.885162,29.648529
1983-04-01,14.628540,14.630737,14.640259,14.638794,14.648315,14.646118,14.644653,14.634399,14.599213,14.572113,...,35.146057,35.302826,35.376831,35.472809,35.836182,35.968781,36.095520,36.228119,36.212738,36.001740
1983-05-01,15.532562,15.540619,15.544281,15.534760,15.519379,15.485687,15.452728,15.417542,15.421204,15.431458,...,39.802551,39.872864,39.891907,39.982025,40.335876,40.441376,40.583496,40.753479,40.825989,40.698517
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-04-01,15.591187,15.575806,15.579468,15.593384,15.587524,15.546478,15.503265,15.460785,15.425598,15.390442,...,33.659576,33.763611,33.816376,33.910126,34.331390,34.477905,34.706482,34.952637,35.095520,35.050079
2022-05-01,16.866669,16.874725,16.884247,16.868866,16.884247,16.892303,16.906219,16.936981,16.931122,16.931122,...,37.923401,37.968079,37.976135,38.023010,38.357086,38.460388,38.623016,38.815704,38.833282,38.644287
2022-06-01,18.165588,18.173645,18.191223,18.210266,18.227844,18.224182,18.227844,18.263031,18.309906,18.326019,...,40.677277,40.675812,40.671417,40.735870,40.985718,41.050171,41.146149,41.165924,41.015015,40.730011


In [94]:
df.columns = [str(i) for i in range(4824)]
df

,0,1,2,3,4,5,6,7,8,9,...,4814,4815,4816,4817,4818,4819,4820,4821,4822,4823
Dates,,,,,,,,,,,,,,,,,,,,,
1983-01-01,15.295197,15.289337,15.287140,15.273224,15.265900,15.260040,15.263702,15.256378,15.265900,15.262238,...,19.656433,19.738495,19.847656,20.000031,20.394928,20.550964,20.638885,20.820557,20.902618,20.652802
1983-02-01,14.360382,14.342804,14.326691,14.287872,14.239502,14.186768,14.151611,14.134003,14.123749,14.110565,...,23.770050,23.813293,23.922455,24.049927,24.530518,24.813293,24.965668,25.231628,25.364227,25.159088
1983-03-01,15.021942,15.041718,15.060760,15.062958,15.060760,15.049042,15.035858,14.998505,14.943542,14.928162,...,28.934235,29.037537,29.131317,29.209717,29.554749,29.629486,29.736450,29.882965,29.885162,29.648529
1983-04-01,14.628540,14.630737,14.640259,14.638794,14.648315,14.646118,14.644653,14.634399,14.599213,14.572113,...,35.146057,35.302826,35.376831,35.472809,35.836182,35.968781,36.095520,36.228119,36.212738,36.001740
1983-05-01,15.532562,15.540619,15.544281,15.534760,15.519379,15.485687,15.452728,15.417542,15.421204,15.431458,...,39.802551,39.872864,39.891907,39.982025,40.335876,40.441376,40.583496,40.753479,40.825989,40.698517
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-04-01,15.591187,15.575806,15.579468,15.593384,15.587524,15.546478,15.503265,15.460785,15.425598,15.390442,...,33.659576,33.763611,33.816376,33.910126,34.331390,34.477905,34.706482,34.952637,35.095520,35.050079
2022-05-01,16.866669,16.874725,16.884247,16.868866,16.884247,16.892303,16.906219,16.936981,16.931122,16.931122,...,37.923401,37.968079,37.976135,38.023010,38.357086,38.460388,38.623016,38.815704,38.833282,38.644287
2022-06-01,18.165588,18.173645,18.191223,18.210266,18.227844,18.224182,18.227844,18.263031,18.309906,18.326019,...,40.677277,40.675812,40.671417,40.735870,40.985718,41.050171,41.146149,41.165924,41.015015,40.730011


In [95]:
df.to_csv("tmax_pixels.csv")